# medcalc-kit — quick start

Load MedCalc-Bench-Verified, inspect one example, compute the reference answer step by step, build the five probes and score answers.

In [1]:
import medcalc_kit as mk
import pandas as pd
pd.set_option('display.max_colwidth', 100)

REVISION = '58fc3de568bfc53a68f32327e58f60dc6eaa17f8'   # pin the dataset version
ds = mk.data.load(REVISION, equation_only=True)
ds

MedCalcDataset(8532 examples, 33 calculators; train: 7872, test: 660)

In [2]:
ds.calculators()

,Calculator ID,Calculator Name,Category,test,train
0,2,Creatinine Clearance (Cockcroft-Gault Equation),lab test,20,131
1,3,CKD-EPI Equations for Glomerular Filtration Rate,lab test,20,483
2,5,Mean Arterial Pressure (MAP),physical,20,927
3,6,Body Mass Index (BMI),physical,20,496
4,7,Calcium Correction for Hypoalbuminemia,lab test,20,185
5,9,MDRD GFR Equation,lab test,20,275
6,10,Ideal Body Weight,physical,20,599
7,11,QTc Bazett Calculator,physical,20,80
8,19,Fibrosis-4 (FIB-4) Index for Liver Fibrosis,lab test,20,90
9,22,Maintenance Fluids Calculations,physical,20,842


## One example

In [3]:
ex = ds.filter(calculator='2', split='test')[0]
print(ex.calculator_name)
print(ex.note[:400], '...')
print('entities:', ex.entities)
print('ground truth:', ex.ground_truth, (ex.lower, ex.upper))

Creatinine Clearance (Cockcroft-Gault Equation)
An 87-year-old man was admitted to our hospital for anorexia for several days, high-grade fever from the previous day, and liver dysfunction. Of note, he had a history of hypertension, diabetes mellitus (DM), and angina. Physical examination findings included: clear consciousness; height, 163 cm; weight, 48 kg; blood pressure, 66/40 mmHg; heart rate, 75/min; respiratory rate, 22/min; oxygen satura ...
entities: {'sex': 'Male', 'age': [87, 'years'], 'weight': [48.0, 'kg'], 'height': [163.0, 'cm'], 'creatinine': [1.4, 'mg/dL']}
ground truth: 25.2381 (23.97619, 26.50001)


## Reference computation
The registry converts the inputs to canonical units, resolves the branch that applies to this patient and computes each step.

In [4]:
d = ex.derive()
print('inputs:', d.inputs)
print('branches:', d.branches)
d.to_frame()

inputs: {'age': 87.0, 'weight': 48.0, 'height': 163.0, 'scr': 1.4}
branches: ['ibw:male', 'bmi<18.5:actual']


,kind,name,expr,value
0,input,age,,87.000000
1,input,weight,,48.000000
2,input,height,,163.000000
3,input,scr,,1.400000
4,coefficient,k_sex,,1.000000
5,step,bmi,weight / (height / 100)^2,18.066167
6,step,height_in,height / 2.54,64.173228
7,step,ibw,50 + 2.3 * (height_in - 60),59.598425
8,step,dosing_weight,weight,48.000000
9,step,crcl,((140 - age) * dosing_weight * k_sex) / (72 * scr),25.238095


## The five probes
Each probe module has `build(example)` (prompt + reference) and `score(item, text)`.

In [5]:
for name, probe in mk.probes.PROBES.items():
    print(f'{name:4} {probe.title:13} {probe.measures}')

E    Extraction    finding the input values (with units) in the patient note
K    Knowledge     writing the formula (general, or for this patient) without computing
S    Substitution  plugging the values into a given formula, without computing
A    Arithmetic    computing a fully written numerical expression
E2E  End to end    the original benchmark task


In [6]:
# K is scored on other real patients of the same calculator: take them from a leakage-free train pool
pool = mk.data.usable_train_pool(ds)
others = pool[pool['Calculator ID'] == ex.calculator_id].sample(50, random_state=0)['entities'].tolist()
item = mk.probes.knowledge.build(ex, point_pool=others)
print(item['messages'][-1]['content'])

Task: What is the patient's Creatinine Clearance using the Cockroft-Gault Equation in terms of mL/min? You should use the patient's adjusted body weight in kg instead of the patient's actual body weight if the patient is overweight or obese based on their BMI. If the patient's BMI's normal, set their adjusted body weight to the minimum of the ideal body and actual weight. If the patient is underweight, please set their adjusted body weight to their actual body weight.

You are NOT given the patient's measurements. Write the general formula needed to answer this task, using only the variables listed below. Their units are fixed as stated; include any unit conversion inside your formula.

Known patient characteristics:
- sex: Male

Variables:
- age: patient age (years)
- weight: body weight (kg)
- height: height (cm)
- scr: serum creatinine (mg/dL)

If the formula depends on thresholds or categories, give one case per branch, each with a condition written with these variables (for exampl

Scoring two candidate answers for K: the classic Cockcroft-Gault with actual weight, and the benchmark's rule (dosing weight depending on BMI). The first is right for this (underweight) patient but wrong for other patients.

Cases must not overlap (here: `< 18.5`, `>= 18.5 and <= 24.9`, otherwise): when several conditions hold at a point, they must give the same value.

In [7]:
naive = '{"cases": [{"condition": "always", "formula": "(140 - age) * weight / (72 * scr)"}]}'
ibw = '50 + 2.3 * (height / 2.54 - 60)'
bmi = 'weight / (height / 100)^2'
full = ('{"cases": ['
        f'{{"condition": "{bmi} < 18.5", "formula": "(140 - age) * weight / (72 * scr)"}},'
        f'{{"condition": "{bmi} >= 18.5 and {bmi} <= 24.9", "formula": "(140 - age) * min({ibw}, weight) / (72 * scr)"}},'
        f'{{"condition": "otherwise", "formula": "(140 - age) * ({ibw} + 0.4 * (weight - ({ibw}))) / (72 * scr)"}}]}}')
print('naive:', mk.probes.score(item, naive))
print('full :', mk.probes.score(item, full))

naive: {'format_valid': True, 'correct': False, 'error_type': 'wrong_elsewhere', 'point_pass_rate': 0.5294117647058824, 'original_point_correct': True, 'n_points': 51}
full : {'format_valid': True, 'correct': True, 'error_type': None, 'point_pass_rate': 1.0, 'original_point_correct': True, 'n_points': 51}


In [8]:
v = mk.probes.knowledge.score(item, naive, details=True)
pd.DataFrame(v['points']).head(8).round(2)

,age,weight,height,scr,registry,model,ok
0,87.0,48.0,163.0,1.40,25.24,25.24,True
1,78.0,60.0,166.0,1.30,39.74,39.74,True
2,61.0,50.0,157.0,0.67,81.88,81.88,True
3,16.0,56.5,155.1,0.63,143.37,154.45,False
4,12.0,48.0,157.5,0.70,121.90,121.90,True
5,22.0,51.0,174.0,0.60,139.31,139.31,True
6,23.0,48.0,164.0,2.47,31.59,31.59,True
7,84.0,80.0,176.0,1.05,55.42,59.26,False


## Substitution and arithmetic

In [9]:
s = mk.probes.substitution.build(ex)
print(s['messages'][-1]['content'])
gold = s['reference']['gold_expression']
print(mk.probes.score(s, '{"expression": "%s"}' % gold))
print(mk.probes.score(s, '{"expression": "%s"}' % round(s['reference']['gold_value'], 2)))   # computed, not substituted

Formula for Creatinine Clearance (Cockcroft-Gault Equation):
crcl = (140 - age) * weight * 1 / (72 * scr)

Variable values:
- age = 87   (patient age (years))
- weight = 48   (body weight (kg))
- scr = 1.4   (serum creatinine (mg/dL))

Substitute the values into the formula. Do NOT compute or simplify anything: copy the formula exactly, replacing each variable by its value.

End your answer with a JSON object of the form {"expression": "<formula with the values substituted>"}
{'format_valid': True, 'correct': True, 'error_type': None, 'value_matches': True, 'missing_variables': [], 'match': 'canonical'}
{'format_valid': True, 'correct': False, 'error_type': 'premature_computation', 'value_matches': True, 'missing_variables': ['age', 'scr', 'weight'], 'match': None}


In [10]:
a = mk.probes.arithmetic.build(ex)
print(a['reference']['expression'], '=', a['reference']['gold_value'])
print(mk.probes.score(a, '{"result": 25.24}'))

(140 - 87) * 48 * 1 / (72 * 1.4) = 25.238095238095237
{'format_valid': True, 'correct': True, 'correct_strict': True, 'error_type': None, 'rel_error': 7.547169811317913e-05}


## Many examples at once
`ProbeSuite` builds every probe for a set of examples. Use the whole dataset as `schema` so that E asks the same fields for every example of a calculator, and a leakage-free pool of other patients as `k_pool` for the K test points.

In [11]:
suite = mk.probes.ProbeSuite(schema=ds, k_pool=pool)
items = suite.build(ds.filter(split='test'))
items.probe.value_counts(sort=False)

probe
E      660
K      660
S      660
A      660
E2E    660
Name: count, dtype: int64

## Running a model
With the optional Hugging Face backend (`pip install medcalc-kit[hf]`) and a GPU:
```python
from medcalc_kit.generation import read_records, run_items
from medcalc_kit.generation.hf import HFBackend
model = HFBackend('Qwen/Qwen2.5-7B-Instruct', device='cuda:0')
run_items(items, 'outputs.jsonl', None, model.model_id, generate_batch=model.generate_batch, batch_size=16)
df = mk.evaluate.score_records('outputs.jsonl', items, condition={'model': 'qwen2.5-7b'})
mk.evaluate.summary(df)
```